# 运行训练与指标解读

本节将展示如何启动 Wordle GRPO 训练，并解读训练过程中产生的关键指标。

> **前提**：已完成第 1 章环境搭建（verl + vllm-ascend 已安装并应用 Wordle AgentLoop 补丁），并完成 3.3 节数据准备（`data/wordle_train.parquet` 已生成）。

---

## 1. 启动训练


### 1.1 准备模型权重

Wordle RL 训练需要一个已完成 SFT 的 Qwen3-1.7B 模型（能输出 `<guess>word</guess>` 格式）。使用 ModelScope 下载预训练好的 SFT 权重：


In [ ]:
%%bash
cd llm_rl/qwen3_wordle
modelscope download --model PrimeIntellect/Qwen3-1.7B-Wordle-SFT --local_dir models/Qwen3-1.7B-Wordle-SFT


### 1.2 启动训练

以下命令在 notebook 中直接运行（前台执行，训练期间 kernel 会阻塞）。训练单步约 250s，155 步约 10 小时。


In [ ]:
%%bash
cd llm_rl/qwen3_wordle
bash run_qwen3_1.7b_wordle_npu.sh


### 训练脚本关键参数

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `MODEL_PATH` | `./models/Qwen3-1.7B-Wordle-SFT` | SFT 模型权重路径 |
| `TRAIN_BATCH_SIZE` | 64 | 训练 batch size |
| `MAX_PROMPT_LENGTH` | 1024 | prompt 最大 token |
| `MAX_RESPONSE_LENGTH` | 4096 | response 最大 token |
| `ROLLOUT_N` | 8 | 每个 prompt 的并行 rollout 数 |
| `MAX_TURNS` | 6 | Wordle 最大猜词轮次 |
| `ACTOR_LR` | 1e-6 | Actor 学习率 |
| `NGPUS_PER_NODE` | 2 | 训练卡数 |
| `ROLLOUT_TP` | 2 | vLLM tensor parallel |

脚本内置关键超参（如需修改请编辑脚本）：

| 超参 | 值 | 说明 |
|------|-----|------|
| `entropy_coeff` | 0.002 | 熵奖励系数，维持探索 |
| `kl_loss_coef` | 0.001 | KL 散度损失系数 |
| `lr_scheduler_type` | cosine | 余弦退火，防止后期过更新 |
| `min_lr_ratio` | 0.1 | 余弦退火终点的 LR 比例 |
| `lr_warmup_steps_ratio` | 0.03 | LR 热身步数占比 |
| `total_epochs` | 5 | 训练总轮数 |
| `save_freq` | 25 | checkpoint 保存间隔（步）|

---

## 2. 训练日志结构

每个训练 step 会输出一行日志，包含大量指标：

```text
训练日志示例 (step 6)

关键指标:
  actor/entropy:0.6618          <- 策略熵
  actor/kl_loss:0.0087          <- KL 散度
  actor/grad_norm:0.5703        <- 梯度范数
  actor/pg_loss:0.0142          <- 策略梯度 loss
  actor/loss:0.0077             <- 总 loss
  critic/score/mean:0.7463      <- 平均奖励
  response_length/mean:1816.15  <- 平均回复长度
  num_turns/mean:5.80           <- 平均交互轮次
  perf/time_per_step:250.00     <- 单步耗时(秒)
```

---

## 3. 验证指标

每 5 步（`test_freq=5`）会进行一次验证，输出 Wordle 特有指标：

```text
验证指标

val-core/wordle/reward/mean@1      <- 验证集平均总奖励
val-aux/wordle/correct/mean@1      <- 验证集猜中率
val-aux/wordle/partial/mean@1      <- 验证集部分匹配分
val-aux/wordle/length_bonus/mean@1 <- 验证集步数奖励
val-aux/wordle/format/mean@1       <- 验证集格式正确率
val-aux/wordle/num_guesses/mean@1  <- 验证集平均猜测次数
```

---

## 4. 训练曲线解读

以下是 Qwen3-1.7B Wordle RL 的典型训练曲线（2 x Ascend 910C）：

### 4.1 Reward 和 Correct 率

| step | reward | correct | 说明 |
|------|--------|---------|------|
| 0 | 0.82 | 15% | 初始水平 |
| 35 | 0.99 | 35% | 快速提升阶段 |
| 75 | 1.03 | 40% | 持续提升 |
| 155 | 1.20 | 70% | 持续上升（cosine 调度）|

### 4.2 Entropy

| step | entropy | 说明 |
|------|---------|------|
| 1 | 0.53 | 初始熵 |
| 50 | 0.35 | 缓慢下降（正常）|
| 100 | 0.28 | 继续下降 |
| 155 | 0.22 | 稳定（未崩塌）|

### 4.3 健康训练的特征

- **reward 上升趋势**：从 0.82 上升到 1.20
- **correct 率提升**：从 15% 提升到 70%
- **entropy 缓慢下降**：从 0.53 降到 0.22，但未崩塌到 0
- **response_length 稳定**：约 1700 token，未全部打满
- **grad_norm 稳定**：0.4-1.0 范围内

> 如果 entropy 突然跌到 0 或 response_length 全部打满 3232，说明训练可能崩塌。详见第 4 章。

---

## 5. 性能参考

| 指标 | 参考值 |
|------|--------|
| 单步耗时 | 约 250s |
| 单步吞吐 | 约 1500 token/s |
| 显存占用 | 约 47GB/卡 |

---

## 6. 训练过程可视化

训练过程中使用 TensorBoard 记录关键指标（reward、entropy、kl_loss 等）。训练日志位于 `llm_rl/qwen3_wordle/checkpoint/` 目录下。

在 notebook 中后台启动 TensorBoard（不阻塞 kernel），通过浏览器访问 `http://<你的IP>:6006` 查看训练曲线：


In [ ]:
%%bash
cd llm_rl/qwen3_wordle
# 后台启动 TensorBoard，日志目录为 checkpoint 下的训练输出
nohup tensorboard --logdir=./checkpoint --bind_all --port 6006 > tensorboard.log 2>&1 &
echo "TensorBoard 已启动，访问 http://<你的IP>:6006"


---

## 课后练习


1. （判断题）验证指标 `val-aux/wordle/correct/mean@1` 表示验证集的猜中率。

2. （判断题）训练日志中的 `actor/entropy` 指标反映了模型的探索能力。

3. （判断题）健康训练中 entropy 应该持续快速下降到 0。

4. （单选题）以下哪个指标可以判断模型是否猜中了秘密单词？
    A. actor/entropy
    B. val-aux/wordle/correct/mean@1
    C. response_length/mean
    D. actor/grad_norm

5. （单选题）训练崩塌的典型信号是什么？
    A. reward 缓慢上升
    B. correct 率从 15% 提升到 45%
    C. entropy 突然跌到 0，response_length 全部打满 3232
    D. entropy 从 0.57 缓慢下降到 0.11

6. （多选题）以下哪些是健康训练的特征？
    A. reward 上升趋势
    B. correct 率提升
    C. entropy 缓慢下降但未崩塌
    D. response_length 稳定，未全部打满

In [ ]:
!cat ../cann-learning-hub/tutorials/rl_training_pipeline/03_wordle_rl_training/answer/03.04_answer.txt